# D2-S02: Euro-area inflation from the OECD

**Was euro-area inflation higher than Sweden's, and how current is that figure?**

Northbridge's largest subcontractor invoices from Sweden, so the manager keeps Swedish and euro-area consumer prices
side by side as background when reading the monthly report. You will request a small piece of the
OECD's consumer-price data, look at what comes back **before** trusting it, turn it into a table, answer the
question, and save the table with a record of where and when it came from.

This is background context. It does not explain Northbridge's costs and it is not a forecast input.

**How this notebook works**
- Work from top to bottom. Click a code cell and press **Shift+Enter**.
- Code cells are labelled **SUPPLIED** (run it), **READ** (run it and read the comments), or **YOUR TURN**.
- Check cells print **OK**, **PROBLEM** or **NOT DONE YET**.
- The kernel (top right) must be the course `.virtual-env-folder` environment.

## Step 0: Set up
**Do:** run the cell. **You should see:** `Course folder found:` followed by your folder.

In [ ]:
# SUPPLIED: course setup. It finds the course folder so the file paths below work wherever the
# course folder is saved. You do not need to read or change this cell.
import os, sys
from pathlib import Path
COURSE_FOLDER = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case_pack").is_dir()), None)
if COURSE_FOLDER is None:
    raise SystemExit("Open the course folder in VS Code (File > Open Folder), then run this cell again.")
os.chdir(COURSE_FOLDER)
sys.path.insert(0, str(COURSE_FOLDER))
print("Course folder found:", COURSE_FOLDER)

## Step 1: What an API is (read, 5 minutes)
An **API** is a way for a program to ask another organisation's system for data, instead of a person downloading a
file from a website. The conversation has two halves:

| Your **request** | The OECD's **response** |
|---|---|
| the **endpoint**: the web address of the dataset | a **status code**: `200` means OK; other numbers mean something went wrong |
| **parameters**: your choices, such as which months | a **content type**: the format of what came back |
| a **header**: the format you would like back | the **body**: the data itself, as structured text |

The OECD answers in **SDMX-JSON**, a structured text format used by statistics agencies. JSON is text made of named
sections inside curly brackets; SDMX adds a fixed layout for statistics. You will look at it in Step 5, but you will
not need to write code that reads it: a tested reader is supplied.

The OECD API is public: it needs no password or key.

## Step 2: Know what the number means before you ask for it (10 minutes)
**Why:** "2.1" is useless unless you know it is a percentage, over what period, measured how.

The series we use is fixed for the course. Its meaning is recorded in
`case_pack/data/external/oecd_data_dictionary.md` (open it from the Explorer). The five fields you need:

| Field | Meaning | Example |
|---|---|---|
| `ref_area_name` | The place measured | Sweden, Euro area |
| `period` | The month measured | 2025-12 |
| `value` | Change in consumer prices compared with the same month one year earlier | 2.1 |
| `unit_label` | What the number is measured in | Percent per annum |
| `retrieved_at_utc` | When the data was downloaded (UTC time), not the month it describes | 2026-09-13T18:33:16Z |

The measure is **HICP**: harmonised consumer prices, compiled the same way in every EU country so places can be
compared. A value of 2.1 means prices were 2.1% higher than a year earlier. It is not EUR 2.10 and not a monthly rise.

## Step 3 (YOUR CHOICE): Build the request
**Why:** you ask only for what you need. A small, fixed request is quicker, easier to check, and easy to explain.

The manager asked about **December 2025**. To see whether December was typical, ask for a few months around it.

**Do:** in the next cell, choose the months. Keep `AREAS` as both areas for the comparison. Months are text in
quotation marks, between `"2024-01"` and `"2026-01"`. Then run the cell.

**You should see:** the endpoint, the three parameters and the number of rows to expect.

In [ ]:
# YOUR CHOICE: which places and months to request.
AREAS = ["SWE", "EA"]      # "SWE" is Sweden, "EA" is the euro area
START = "2025-09"          # first month, as "YYYY-MM"
END = "2026-01"            # last month, as "YYYY-MM"

# SUPPLIED: put the request together from those choices and print it. Nothing is sent yet.
from case_pack.course_tools import oecd
request = oecd.build_request(AREAS, START, END)

## Step 4: Send the request, or use the saved response
**Why:** a live request can fail (no network, the service is busy). The course keeps a response saved on
2026-09-13, so everyone can continue. What matters is being honest about which one you used.

**Do:** run the cell with `MODE = "offline"`. After the instructor's live demonstration you may change it to
`"live"` and run again.

**You should see:** the status code, content type, size and retrieval time. Offline mode says clearly that it is a
saved response.

If a live request fails, the cell says so. Record the message, set `MODE = "offline"` and run the cell again.
Do not keep retrying.

In [ ]:
# YOUR CHOICE: "offline" uses the saved response; "live" sends the request to the OECD now.
MODE = "offline"

# SUPPLIED: get the response and print its status, format, size and retrieval time.
response = oecd.get_response(request, MODE)

## Step 5: Look at the response before converting it
**Why:** never turn data into a table without looking at what arrived. The body could be an error message, the wrong
format, or data with a different meaning.

**Do:** run the cell and read the output from top to bottom. It shows:
1. the first characters of the body, exactly as received;
2. the **structures** section: the meaning (which dimensions exist and their possible values);
3. the **dataSets** section: the numbers, each stored under a key such as `0:0:0:0:0:0:0:0:3`;
4. the first key decoded by hand: each number in the key is a position in one dimension's list of values.

Discuss with your neighbour: why does the body store positions instead of repeating "Euro area" on every number?

In [ ]:
# SUPPLIED: show the start of the body and decode one observation by hand.
oecd.peek(response)

## Step 6: Turn the response into a table and check its meaning
**Why:** the supplied reader decodes every observation the way you just decoded one, and then checks the result:
the right number of rows, one measure only, and no duplicates.

**Do:** run the cell. **You should see:** three **OK** lines and a table with one row per area and month.

In [ ]:
# SUPPLIED: decode the body into a table and check it.
table = oecd.to_table(response, request)
table[oecd.SHORT_COLUMNS] if table is not None else None

## Step 7 (YOUR TURN): Put the two areas side by side
**Why:** the manager's question is a comparison. Each month should be one row, with Sweden and the euro area next to
each other and the difference between them.

**Do:** copy this prompt into Copilot Chat, read the answer, paste it into the next cell and run it.

> I have a pandas DataFrame called `table` with the columns ref_area (either "SWE" or "EA"), period (a month such as
> "2025-12") and value (a number), plus others. Write short, commented pandas code that creates a new DataFrame
> called `gaps` with one row per period and the columns period, SWE, EA and gap, where gap is EA minus SWE, rounded
> to 2 decimals. Do not read or write any files.

The difference between two percentages is measured in **percentage points**. From 2.1% to 2.0% is a gap of
-0.1 percentage points, not -0.1%.

In [ ]:
# YOUR TURN: paste Copilot's code below this line. It must create a DataFrame called `gaps`.

gaps = None

In [ ]:
# SUPPLIED: check YOUR TURN. You should see two OK lines, then your table.
oecd.check_gaps(gaps, table)
gaps

## Step 8: Save the table with its source record
**Why:** a number without its source and retrieval time cannot be checked later, and cached data must never be
passed off as new.

**Do:** run the cell. **You should see:** three files saved in `outputs/D2-S02/`. Open `source_record.json` from the
Explorer and find `data_status` and `retrieved_at_utc`.

In [ ]:
# SUPPLIED: save the table, the untouched response body and a record of where and when it came from.
oecd.save_extract(table, response)

## Step 9: Answer the manager
Discuss with your neighbour, then write your answer in a new notes file: in the Explorer, right-click
`outputs/D2-S02`, choose **New File**, and name it `answer.md`. Use these headings:

```text
Answer (two sentences):
Was December typical of the months around it?
Source, retrieval time, and live or cached:
What this figure does not tell us about Northbridge:
```

**Start again:** choose **Restart** in the notebook toolbar and run from Step 0. Nothing in `case_pack` is changed.

**Optional extension:** set `MODE = "live"` and run Steps 4 to 8 again. Compare the retrieval time and any values
that differ from the saved response. Which would you hand to the manager, and how would you label it?

## Check your result
Read this only after your answer is written.

| Month | Sweden % | Euro area % | Gap (EA minus SWE), percentage points |
|---|---:|---:|---:|
| 2025-11 | 2.2 | 2.1 | -0.1 |
| 2025-12 | 2.1 | 2.0 | -0.1 |
| 2026-01 | 2.0 | 1.7 | -0.3 |

In December 2025 euro-area inflation (2.0%) was slightly **lower** than Sweden's (2.1%). A lower rate still means
prices **rose**, just more slowly. A live request may show revised values: report what you actually retrieved.